In [ ]:
%pip install nbformat

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import plotly.graph_objects as go
import plotly.express as px


# ==============================
# 1. Configuration
# ==============================

BASE_DIR = Path(r"E:\SOPHIAS")

SESSION_ID = "202503111"

events_path = BASE_DIR / SESSION_ID / "AICoFe" / "Logs" / "events.csv"
hr_path = BASE_DIR / SESSION_ID / "Watch-DMLT" / "P_filter" / "HeartRate.csv"

print("Events file:", events_path)
print("Heart rate file:", hr_path)


# ==============================
# 2. Load files
# ==============================

events = pd.read_csv(events_path)
hr = pd.read_csv(hr_path)


# ==============================
# 3. Robust timestamp function
# ==============================

def unix_to_seconds(column):
    """
    Convert UNIX timestamps to seconds.

    This function corrects each value independently:
    - If the value is greater than 1e12, it is considered milliseconds.
    - Otherwise, it is considered seconds.
    """
    x = pd.to_numeric(column, errors="coerce")
    x = x.where(x < 1e12, x / 1000)
    return x


# ==============================
# 4. Normalize timestamps
# ==============================

# Heart rate: the Timestamp column usually comes in milliseconds
hr["time_unix"] = unix_to_seconds(hr["Timestamp"])

# Events: the file may contain a mix of seconds and milliseconds
events["start_unix"] = unix_to_seconds(events["Start_UNIX"])
events["end_unix"] = unix_to_seconds(events["End_UNIX"])

# Remove rows without a valid start time
events = events.dropna(subset=["start_unix", "Key"])
hr = hr.dropna(subset=["time_unix"])


# ==============================
# 5. Select heart rate column
# ==============================

if "HR_filtered" in hr.columns and hr["HR_filtered"].notna().sum() > 0:
    hr_column = "HR_filtered"
else:
    hr_column = "Heart Rate"

hr[hr_column] = pd.to_numeric(hr[hr_column], errors="coerce")
hr = hr.dropna(subset=[hr_column])


# ==============================
# 6. Create relative time axis
# ==============================

# Time zero: beginning of the heart rate recording
t0 = hr["time_unix"].min()

hr["time_rel"] = hr["time_unix"] - t0
events["start_rel"] = events["start_unix"] - t0
events["end_rel"] = events["end_unix"] - t0

hr_min = hr["time_rel"].min()
hr_max = hr["time_rel"].max()


# ==============================
# 7. Separate interval events and point events
# ==============================

# Events with both start and end times
interval_events = events.dropna(subset=["end_rel"]).copy()

# Point events: they have a start time but no end time
point_events = events[events["end_rel"].isna()].copy()

# Keep only events that fall within the heart rate time range
interval_events = interval_events[
    (interval_events["end_rel"] >= hr_min) &
    (interval_events["start_rel"] <= hr_max)
].copy()

point_events = point_events[
    (point_events["start_rel"] >= hr_min) &
    (point_events["start_rel"] <= hr_max)
].copy()

# Clip interval events to the visible heart rate range
interval_events["start_plot"] = interval_events["start_rel"].clip(
    lower=hr_min,
    upper=hr_max
)

interval_events["end_plot"] = interval_events["end_rel"].clip(
    lower=hr_min,
    upper=hr_max
)

print("Heart rate samples:", len(hr))
print("Visible interval events:", len(interval_events))
print("Visible point events:", len(point_events))


# ==============================
# 8. Prepare colors by event label
# ==============================

event_labels = pd.concat([
    interval_events["Key"],
    point_events["Key"]
]).dropna().unique()

# Palette with more vivid colors
palette = (
    px.colors.qualitative.Bold +
    px.colors.qualitative.Vivid +
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Alphabet
)

color_map = {
    label: palette[i % len(palette)]
    for i, label in enumerate(event_labels)
}

# Vertical range of the plot
y_min = hr[hr_column].min()
y_max = hr[hr_column].max()

# Small visual margin
y_margin = (y_max - y_min) * 0.08
y_min_plot = y_min - y_margin
y_max_plot = y_max + y_margin

# Event label opacity
# Increase this to 0.60 if you want the events to be even more visible
event_opacity = 0.55


# ==============================
# 9. Create interactive figure
# ==============================

fig = go.Figure()


# ------------------------------
# Heart rate: always visible
# ------------------------------

fig.add_trace(
    go.Scatter(
        x=hr["time_rel"],
        y=hr[hr_column],
        mode="lines",
        name="Heart rate",
        line=dict(color="black", width=2.5),
        showlegend=False,  # Heart rate is not shown in the legend, so it cannot be hidden
        hovertemplate=(
            "Time: %{x:.2f} s<br>"
            "Heart rate: %{y:.1f}<extra></extra>"
        )
    )
)


# ------------------------------
# Events by label
# ------------------------------

for label in event_labels:

    label_interval_events = interval_events[
        interval_events["Key"] == label
    ]

    label_point_events = point_events[
        point_events["Key"] == label
    ]

    color = color_map[label]

    # Control variable so that each label appears only once in the legend
    legend_added = False


    # ==============================
    # Intervals as rectangles
    # ==============================

    if len(label_interval_events) > 0:
        x_rect = []
        y_rect = []

        for _, row in label_interval_events.iterrows():
            start = row["start_plot"]
            end = row["end_plot"]

            x_rect += [start, end, end, start, start, None]
            y_rect += [y_min_plot, y_min_plot, y_max_plot, y_max_plot, y_min_plot, None]

        fig.add_trace(
            go.Scatter(
                x=x_rect,
                y=y_rect,
                mode="lines",
                fill="toself",
                fillcolor=color,
                line=dict(color=color, width=0),
                opacity=event_opacity,
                name=label,
                legendgroup=label,
                showlegend=True,
                visible="legendonly",  # Hidden by default
                hoverinfo="skip"
            )
        )

        legend_added = True


    # ==============================
    # Point events as vertical lines
    # ==============================

    if len(label_point_events) > 0:
        x_lines = []
        y_lines = []

        for _, row in label_point_events.iterrows():
            t = row["start_rel"]

            x_lines += [t, t, None]
            y_lines += [y_min_plot, y_max_plot, None]

        fig.add_trace(
            go.Scatter(
                x=x_lines,
                y=y_lines,
                mode="lines",
                name=label,
                legendgroup=label,
                showlegend=not legend_added,
                visible="legendonly",  # Hidden by default
                line=dict(
                    color=color,
                    width=2.5,
                    dash="dash"
                ),
                opacity=0.95,
                hovertemplate=(
                    f"Event: {label}<br>"
                    "Time: %{x:.2f} s<extra></extra>"
                )
            )
        )


# ==============================
# 10. Figure configuration
# ==============================

fig.update_layout(
    title=f"Interactive multimodal synchronization: heart rate and events | ID {SESSION_ID}",
    xaxis_title="Time from the beginning of the heart rate recording, in seconds",
    yaxis_title="Heart rate",
    width=1200,
    height=600,
    template="plotly_white",

    legend=dict(
        title="Event labels",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
        groupclick="togglegroup"
    ),

    hovermode="x unified"
)

fig.update_xaxes(range=[hr_min, hr_max])
fig.update_yaxes(range=[y_min_plot, y_max_plot])

fig.show()